# Databricks Unity Catalog Full Permissions Audit Notebook

This notebook builds a unified permissions audit dataset across Unity Catalog objects:
- Catalogs
- Schemas
- Tables / Views
- Functions
- Volumes
- External Locations
- Storage Credentials

Run sequentially to create a governance audit layer.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS governance_audit;

## Create unified Unity Catalog permissions view

In [ ]:
%sql
CREATE OR REPLACE VIEW governance_audit.unity_catalog_permissions AS

SELECT 'CATALOG' AS object_type,
       catalog_name AS catalog,
       NULL AS schema,
       NULL AS object_name,
       privilege_type,
       grantee
FROM system.information_schema.catalog_privileges

UNION ALL

SELECT 'SCHEMA',
       catalog_name,
       schema_name,
       NULL,
       privilege_type,
       grantee
FROM system.information_schema.schema_privileges

UNION ALL

SELECT 'TABLE_OR_VIEW',
       table_catalog,
       table_schema,
       table_name,
       privilege_type,
       grantee
FROM system.information_schema.table_privileges

UNION ALL

SELECT 'FUNCTION',
       routine_catalog,
       routine_schema,
       routine_name,
       privilege_type,
       grantee
FROM system.information_schema.routine_privileges;

## External Location permissions

In [ ]:
%sql
CREATE OR REPLACE VIEW governance_audit.external_location_permissions AS
SELECT 'EXTERNAL_LOCATION' AS object_type,
       external_location_name AS object_name,
       privilege_type,
       grantee
FROM system.information_schema.external_location_privileges;

## Volume permissions

In [ ]:
%sql
CREATE OR REPLACE VIEW governance_audit.volume_permissions AS
SELECT 'VOLUME' AS object_type,
       volume_catalog,
       volume_schema,
       volume_name,
       privilege_type,
       grantee
FROM system.information_schema.volume_privileges;

## Storage credential permissions

In [ ]:
%sql
CREATE OR REPLACE VIEW governance_audit.storage_credential_permissions AS
SELECT 'STORAGE_CREDENTIAL' AS object_type,
       storage_credential_name,
       privilege_type,
       grantee
FROM system.information_schema.storage_credential_privileges;

## Create master flattened audit dataset

In [ ]:
%sql
CREATE OR REPLACE VIEW governance_audit.full_permissions_audit AS

SELECT * FROM governance_audit.unity_catalog_permissions

UNION ALL

SELECT object_type,
       NULL,
       NULL,
       object_name,
       privilege_type,
       grantee
FROM governance_audit.external_location_permissions

UNION ALL

SELECT object_type,
       volume_catalog,
       volume_schema,
       volume_name,
       privilege_type,
       grantee
FROM governance_audit.volume_permissions

UNION ALL

SELECT object_type,
       NULL,
       NULL,
       storage_credential_name,
       privilege_type,
       grantee
FROM governance_audit.storage_credential_permissions;

## Query audit dataset

In [ ]:
%sql
SELECT *
FROM governance_audit.full_permissions_audit
ORDER BY object_type, catalog, schema, object_name;